## Desafio - Aula 6.2 (Regressão Linear)

In [9]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression

dados = fetch_california_housing()
X, y = dados.data, dados.target

In [10]:
class MinimosQuadrados:
    def __init__(self,  metodo='pinv'):
        # Validar método e guardar self.metodo
        try:
            if metodo in ['inv', 'solve', 'pinv']:
                self.metodo = metodo
            else:
                raise ValueError(f"O método {metodo} está fora do escopo")
        except ValueError as e:
            print(f"Método fora do escopo! {e}")
            self.metodo = 'pinv' # Padrão

    def adicionar_intercepto(self, X):
        # Devolver X como uma coluna de 1s à esquerda
        uns = np.ones((X.shape[0], 1))
        return np.hstack((uns, X))
    
    def fit(self, X, y):
        # Montar Xb e calcular self.coef_
        Xb = self.adicionar_intercepto(X)

        if self.metodo == 'inv':
            self.coef_ = np.linalg.inv(Xb.T @ Xb) @ Xb.T @ y
        elif self.metodo == 'solve':
            self.coef_ = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)
        elif self.metodo == 'pinv':
            self.coef_ = np.linalg.pinv(Xb) @ y

        return self

        
    def predict(self, X):
        # Mesmo tratamento do fit, depois Xb @ self.coef_

        Xb = self.adicionar_intercepto(X)
        # Desnecessário repetir validação, mas exercício pede tratamento
        """
        if self.metodo == 'inv':
            y_hat = Xb @ self.coef_
        elif self.metodo == 'solve':
            y_hat = Xb @ self.coef_
        elif self.metodo == 'pinv':
            y_hat = Xb @ self.coef_
        """
        y_hat = Xb @ self.coef_
        return y_hat
    
    def score(self, X, y):
        # R² = 1 - SQres / SQtot

        y_hat = self.predict(X)

        # Soma dos Quadrados dos Resíduos (SQres)
        SQres = np.sum((y - y_hat) ** 2)
        # Soma dos Quadrados Total (SQtot)
        SQtot = np.sum((y - np.mean(y)) ** 2)

        r2 = 1 - SQres / SQtot

        return r2
    

## Teste

In [11]:
n = X.shape[0]
Xb = np.column_stack([np.ones(n), X])
beta_pinv = np.linalg.pinv(Xb) @ y
y_hat = Xb @ beta_pinv

r2_aula = 1 - np.sum((y - y_hat)**2) / np.sum((y - y.mean())**2)
print(r2_aula)

0.606232685199805


In [15]:
# Teste e comparação com a classe MinimosQuadrados
m = MinimosQuadrados(metodo="pinv").fit(X,y)
lr = LinearRegression().fit(X,y)

np.allclose(m.coef_, np.r_[lr.intercept_, lr.coef_])
np.allclose(m.predict(X), lr.predict(X))
np.isclose(m.score(X,y), lr.score(X,y))

X_dup = np.column_stack([X, X[:, 0]])
np.linalg.cond(m.adicionar_intercepto(X))
np.linalg.cond(m.adicionar_intercepto(X_dup))
# MinimosQuadrados(metodo="inv").fit(X_dup, y)

print("Condição original:", np.linalg.cond(m.adicionar_intercepto(X)))
print("Condição com duplicada:", np.linalg.cond(m.adicionar_intercepto(X_dup)))

# Testando 'inv' (espera-se o erro)
try:
    MinimosQuadrados(metodo="inv").fit(X_dup, y)
except np.linalg.LinAlgError as e:
    print(f"Erro esperado com 'inv': {e}")

# Testando 'pinv' (deve funcionar)
m_dup = MinimosQuadrados(metodo="pinv").fit(X_dup, y)
print("Sucesso com 'pinv'! Coeficientes calculados:", len(m_dup.coef_))


Condição original: 238335.7960607902
Condição com duplicada: 2.1750205535073219e+18
Erro esperado com 'inv': Singular matrix
Sucesso com 'pinv'! Coeficientes calculados: 10
